# AnimalCLEF2026 Salamander EVA02/CLIP + ALIKED Merge-Only v20260430

This notebook is a **Salamander-only incremental branch** over the current best public-LB partition:

```text
0.32903 = 0.32684 final Texas boundary split-only base
        + six ultra-strict Lynx ALIKED merge-only fixes
```

It keeps Lynx, SeaTurtle, and Texas unchanged, then tries to repair over-split Salamander clusters using the missing piece from the 2025 winning-style recipe:

- original Salamander view
- orientation-normalized Salamander view: right -> rotate -90, left -> rotate +90, bottom -> 180, top unchanged
- EVA02/CLIP 5-crop global descriptors
- ALIKED local descriptors with geometric verification
- train-identity calibrated pair probabilities
- merge-only graph updates with max cluster caps

## Required Kaggle Inputs

Attach:

1. `animal-clef-2026`
2. output of `hanifnoerrofiq/lb-0-32903-lynxaliked-mergeonly-reproducer-v20260430`, or the equivalent LB032903 reproducer notebook output

If the LB032903 reproducer output is not attached, this notebook can rebuild the same base from:

- output of `hanifnoerrofiq/final-texas-boundary-split-only`
- output of `hanifnoerrofiq/lynx-low-light-eva-clip-aliked`

## Runtime

Use GPU. T4 should work. This notebook does not recompute SAM segmentation.

## Output

Default first candidate:

```text
/kaggle/working/animalclef_salamander_eva_aliked_mergeonly_v20260430/submission.csv
```

Profile files are also written under:

```text
/kaggle/working/animalclef_salamander_eva_aliked_mergeonly_v20260430/submissions/
```

Do not blindly submit a profile that makes many Salamander merges. Inspect `selected_rules`, action count, and shape report first.
            

In [ ]:
from pathlib import Path

SCRIPT_TEXT = "#!/usr/bin/env python3\n\"\"\"\nAnimalCLEF2026 Salamander EVA02/CLIP + ALIKED merge-only branch.\n\nThis is a Salamander-only incremental postprocessor over the current best\n0.32903 partition. It keeps Lynx, SeaTurtle, and Texas unchanged, then tries\nto recover over-split Salamander identities using the part of the 2025-winning\nrecipe we have not yet used for Salamander:\n\n* original and orientation-normalized Salamander views\n* EVA02/CLIP 5-crop global descriptors\n* ALIKED local feature matching with geometric verification\n* train-identity calibrated pair probabilities\n* merge-only graph postprocessing with max cluster-size caps\n\nIf the 0.32903 current-best submission is not attached, this script can rebuild\nit from the final-texas-boundary-split-only 0.32684 base plus the Lynx\nEVA/ALIKED pair-score report.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport math\nimport os\nimport pickle\nimport random\nimport subprocess\nimport sys\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFile, ImageOps\nfrom tqdm.auto import tqdm\n\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nVERSION = \"salamander_eva_aliked_mergeonly_v20260430\"\nSALAMANDER = \"SalamanderID2025\"\nLYNX = \"LynxID2025\"\nSEED = 20260430\nPAD = np.array([128, 128, 128], dtype=np.uint8)\n\n\nCURRENT_BEST_NAMES = [\n    \"submission_032684_lynx_aliked_prob098_mergeonly_v20260430.csv\",\n    \"submission_lb032903_lynx_aliked_prob098_mergeonly_v20260430.csv\",\n    \"submission.csv\",\n]\nBASE_032684_NAMES = [\n    \"submission_final_texas_boundary_splitonly_balanced_from_032583_v20260430.csv\",\n    \"submission.csv\",\n]\nLYNX_PAIR_SCORE_NAME = \"lynx_lowlight_eva_aliked_v20260430_test_pair_scores.csv\"\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--current-best\", type=str, default=None)\n    parser.add_argument(\"--base-032684\", type=str, default=None)\n    parser.add_argument(\"--lynx-pair-scores\", type=str, default=None)\n    parser.add_argument(\"--output-root\", type=str, default=f\"/kaggle/working/animalclef_{VERSION}\")\n    parser.add_argument(\"--img-size\", type=int, default=512)\n    parser.add_argument(\"--top-k\", type=int, default=18)\n    parser.add_argument(\"--train-pair-limit\", type=int, default=7000)\n    parser.add_argument(\"--test-pair-limit\", type=int, default=14000)\n    parser.add_argument(\"--max-aliked-keypoints\", type=int, default=512)\n    parser.add_argument(\"--aliked-match-threshold\", type=float, default=0.72)\n    parser.add_argument(\"--eva-batch-size\", type=int, default=32)\n    parser.add_argument(\"--eva-model\", type=str, default=\"EVA02-B-16\")\n    parser.add_argument(\"--eva-pretrained\", type=str, default=\"merged2b_s8b_b131k\")\n    parser.add_argument(\"--disable-eva\", action=\"store_true\")\n    parser.add_argument(\"--disable-aliked\", action=\"store_true\")\n    parser.add_argument(\"--allow-orb-fallback\", action=\"store_true\")\n    parser.add_argument(\"--save-visualizations\", action=\"store_true\")\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef seed_everything(seed: int = SEED) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n\n\ndef install_if_missing(import_name: str, pip_name: str) -> bool:\n    try:\n        __import__(import_name)\n        return True\n    except Exception:\n        pass\n    try:\n        print(f\"[deps] installing {pip_name}\")\n        subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", pip_name], check=True)\n        __import__(import_name)\n        return True\n    except Exception as exc:\n        print(f\"[deps] {pip_name} unavailable: {exc}\")\n        return False\n\n\ndef candidate_roots() -> list[Path]:\n    roots: list[Path] = []\n    for value in [\n        os.environ.get(\"DATA_ROOT\"),\n        \"/kaggle/input/competitions/animal-clef-2026\",\n        \"/kaggle/input/animal-clef-2026\",\n        \"/kaggle/input\",\n        \"/kaggle/working\",\n        \"C:/Users/Hanif/Documents/kaggle/AnimalCLEF2026/animal-clef-2026\",\n        \"C:/Users/Hanif/Documents/kaggle/AnimalCLEF2026/current_wildfusion_graph_v20260423\",\n        \"C:/Users/Hanif/Documents/kaggle/AnimalCLEF2026/kaggle_output_lynx_lowlight_eva_aliked_v315501274\",\n        \"C:/Users/Hanif/Documents/kaggle/AnimalCLEF2026\",\n        \".\",\n    ]:\n        if value:\n            p = Path(value)\n            if p.exists():\n                roots.append(p)\n    out = []\n    seen = set()\n    for root in roots:\n        try:\n            key = str(root.resolve())\n        except Exception:\n            key = str(root)\n        if key not in seen:\n            out.append(root)\n            seen.add(key)\n    return out\n\n\ndef find_data_root(arg: str | None) -> Path:\n    if arg:\n        p = Path(arg)\n        if (p / \"metadata.csv\").exists() and (p / \"sample_submission.csv\").exists():\n            return p.resolve()\n        raise FileNotFoundError(f\"--data-root is not an AnimalCLEF root: {p}\")\n    for root in candidate_roots():\n        if (root / \"metadata.csv\").exists() and (root / \"sample_submission.csv\").exists():\n            return root.resolve()\n    for root in candidate_roots():\n        try:\n            for meta in root.rglob(\"metadata.csv\"):\n                if (meta.parent / \"sample_submission.csv\").exists():\n                    return meta.parent.resolve()\n        except Exception:\n            pass\n    raise FileNotFoundError(\"Could not locate AnimalCLEF2026 data root.\")\n\n\ndef rank_current_best(path: Path) -> tuple[int, int, str]:\n    text = str(path).replace(\"\\\\\", \"/\").lower()\n    penalty = 0\n    if path.name == \"submission.csv\" and not any(\n        token in text\n        for token in [\n            \"lb032903\",\n            \"0-32903\",\n            \"0_32903\",\n            \"lynx_aliked_reproducer\",\n            \"lynxaliked-merge\",\n        ]\n    ):\n        penalty += 1000\n    if \"local_smoke\" in text or \"__pycache__\" in text:\n        penalty += 100\n    if \"032684_lynx_aliked\" in path.name.lower() or \"lb032903\" in text:\n        penalty -= 80\n    if \"animalclef_lb032903_lynx_aliked_reproducer\" in text:\n        penalty -= 70\n    if \"current_wildfusion_graph\" in text:\n        penalty -= 20\n    if \"/kaggle/input/\" in text:\n        penalty -= 10\n    if path.name == \"submission.csv\":\n        penalty += 10\n    return penalty, len(text), text\n\n\ndef rank_base_032684(path: Path) -> tuple[int, int, str]:\n    text = str(path).replace(\"\\\\\", \"/\").lower()\n    penalty = 0\n    if \"local_smoke\" in text or \"__pycache__\" in text:\n        penalty += 100\n    if \"final-texas-boundary-split-only\" in text or \"animalclef_texas_boundary_splitonly_final\" in text:\n        penalty -= 60\n    if \"submission_final_texas_boundary_splitonly\" in path.name.lower():\n        penalty -= 40\n    if \"/kaggle/input/\" in text:\n        penalty -= 10\n    if path.name == \"submission.csv\":\n        penalty += 8\n    return penalty, len(text), text\n\n\ndef rank_pair_path(path: Path) -> tuple[int, int, str]:\n    text = str(path).replace(\"\\\\\", \"/\").lower()\n    penalty = 0\n    if \"local_smoke\" in text or \"__pycache__\" in text:\n        penalty += 100\n    if \"lynx-low-light-eva-clip-aliked\" in text or \"animalclef_lynx_lowlight_eva_aliked\" in text:\n        penalty -= 60\n    if \"/reports/\" in text:\n        penalty -= 10\n    if \"/kaggle/input/\" in text:\n        penalty -= 10\n    return penalty, len(text), text\n\n\ndef find_optional_file(names: Iterable[str], arg: str | None, ranker) -> Path | None:\n    if arg:\n        p = Path(arg)\n        if p.exists():\n            return p.resolve()\n        raise FileNotFoundError(f\"Path does not exist: {p}\")\n    hits = []\n    for root in candidate_roots():\n        for name in names:\n            direct = root / name\n            if direct.exists():\n                hits.append(direct)\n            try:\n                hits.extend(root.rglob(name))\n            except Exception:\n                pass\n    hits = sorted({p.resolve() for p in hits if p.exists()}, key=ranker)\n    return hits[0] if hits else None\n\n\ndef find_file(names: Iterable[str], arg: str | None, ranker) -> Path:\n    hit = find_optional_file(names, arg, ranker)\n    if hit is None:\n        raise FileNotFoundError(f\"Could not find any of: {list(names)}\")\n    return hit\n\n\ndef load_submission(path: Path) -> pd.DataFrame:\n    df = pd.read_csv(path)\n    if \"image_id\" not in df.columns or \"cluster\" not in df.columns:\n        raise ValueError(f\"{path} must contain image_id and cluster columns.\")\n    df = df[[\"image_id\", \"cluster\"]].copy()\n    df[\"image_id\"] = df[\"image_id\"].astype(int)\n    df[\"cluster\"] = df[\"cluster\"].astype(str)\n    if df[\"image_id\"].duplicated().any():\n        raise ValueError(f\"Duplicate image_id values in {path}\")\n    return df\n\n\ndef validate_submission(sub: pd.DataFrame, sample: pd.DataFrame) -> None:\n    if list(sub.columns) != [\"image_id\", \"cluster\"]:\n        raise ValueError(\"Submission columns must be exactly image_id, cluster.\")\n    if len(sub) != len(sample):\n        raise ValueError(f\"Submission rows {len(sub)} != sample rows {len(sample)}\")\n    if sub[\"image_id\"].astype(int).tolist() != sample[\"image_id\"].astype(int).tolist():\n        raise ValueError(\"Submission image order does not match sample_submission.csv\")\n    if sub[\"cluster\"].isna().any():\n        raise ValueError(\"Submission contains null cluster labels.\")\n    max_len = int(sub[\"cluster\"].astype(str).str.len().max())\n    if max_len > 64:\n        raise ValueError(f\"Cluster labels too long: max length {max_len}\")\n\n\nclass UF:\n    def __init__(self, values: Iterable[int]):\n        self.parent = {int(v): int(v) for v in values}\n        self.size = {int(v): 1 for v in values}\n\n    def find(self, value: int) -> int:\n        value = int(value)\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a: int, b: int) -> bool:\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return False\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n        return True\n\n\ndef compact_species_labels(sub: pd.DataFrame, sample: pd.DataFrame, test_rows: pd.DataFrame) -> pd.DataFrame:\n    parts = []\n    sample_ids = set(sample[\"image_id\"].astype(int))\n    for species, group in test_rows[test_rows[\"image_id\"].isin(sample_ids)].groupby(\"dataset\", sort=True):\n        ids = set(group[\"image_id\"].astype(int))\n        part = sub[sub[\"image_id\"].astype(int).isin(ids)].copy()\n        groups = []\n        for _, members in part.groupby(\"cluster\"):\n            groups.append(sorted(members[\"image_id\"].astype(int).tolist()))\n        labels = {}\n        for idx, members in enumerate(sorted(groups, key=lambda xs: (min(xs), len(xs), max(xs)))):\n            for image_id in members:\n                labels[image_id] = f\"cluster_{species}_{idx}\"\n        parts.append(pd.DataFrame({\"image_id\": sorted(ids), \"cluster\": [labels[i] for i in sorted(ids)]}))\n    final = pd.concat(parts, ignore_index=True)\n    final = sample[[\"image_id\"]].merge(final, on=\"image_id\", how=\"left\")\n    return final[[\"image_id\", \"cluster\"]]\n\n\ndef build_032903_from_components(\n    base_032684: pd.DataFrame,\n    lynx_pairs: pd.DataFrame,\n    sample: pd.DataFrame,\n    test_rows: pd.DataFrame,\n) -> pd.DataFrame:\n    lynx_ids = (\n        test_rows[test_rows[\"dataset\"].eq(LYNX) & test_rows[\"image_id\"].isin(sample[\"image_id\"])]\n        [\"image_id\"]\n        .astype(int)\n        .tolist()\n    )\n    current = base_032684[base_032684[\"image_id\"].astype(int).isin(lynx_ids)].copy()\n    uf = UF(lynx_ids)\n    for _, group in current.groupby(\"cluster\"):\n        ids = group[\"image_id\"].astype(int).tolist()\n        for a, b in itertools.combinations(ids, 2):\n            uf.union(a, b)\n\n    same_current = bool_series(lynx_pairs[\"same_current_cluster\"])\n    accepted = lynx_pairs[\n        (~same_current)\n        & (~lynx_pairs[\"orientation_relation\"].astype(str).eq(\"opposite_flank\"))\n        & (lynx_pairs[\"local_score\"].astype(float) >= 0.96)\n        & (lynx_pairs[\"inliers\"].astype(float) >= 50)\n        & (lynx_pairs[\"match_prob\"].astype(float) >= 0.98)\n    ].copy()\n    accepted = accepted.sort_values([\"local_score\", \"inliers\", \"match_prob\"], ascending=False)\n    for row in accepted.itertuples(index=False):\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        if a not in uf.parent or b not in uf.parent or uf.find(a) == uf.find(b):\n            continue\n        size_after = uf.size[uf.find(a)] + uf.size[uf.find(b)]\n        if size_after <= 84:\n            uf.union(a, b)\n\n    groups = {}\n    for image_id in lynx_ids:\n        groups.setdefault(uf.find(image_id), []).append(image_id)\n    labels = {}\n    for idx, members in enumerate(sorted(groups.values(), key=lambda xs: (min(xs), len(xs), max(xs)))):\n        for image_id in members:\n            labels[image_id] = f\"cluster_{LYNX}_{idx}\"\n    out = base_032684.copy()\n    mask = out[\"image_id\"].astype(int).isin(labels)\n    out.loc[mask, \"cluster\"] = out.loc[mask, \"image_id\"].astype(int).map(labels)\n    return compact_species_labels(out, sample, test_rows)\n\n\ndef bool_series(s: pd.Series) -> pd.Series:\n    if s.dtype == bool:\n        return s\n    return s.astype(str).str.lower().isin([\"true\", \"1\", \"yes\"])\n\n\ndef prepare_metadata(data_root: Path) -> pd.DataFrame:\n    metadata = pd.read_csv(data_root / \"metadata.csv\").copy()\n    if \"dataset\" not in metadata.columns:\n        metadata[\"dataset\"] = metadata[\"path\"].astype(str).str.replace(\"\\\\\", \"/\", regex=False).str.split(\"/\").str[1]\n    if \"split\" not in metadata.columns:\n        metadata[\"split\"] = np.where(\n            metadata[\"path\"].astype(str).str.replace(\"\\\\\", \"/\", regex=False).str.contains(\"/test/\"),\n            \"test\",\n            \"train\",\n        )\n    metadata[\"image_id\"] = metadata[\"image_id\"].astype(int)\n    metadata[\"abs_path\"] = metadata[\"path\"].apply(lambda p: str((data_root / str(p)).resolve()))\n    return metadata\n\n\ndef orientation_normalize(img: Image.Image, orientation: str) -> Image.Image:\n    orientation = str(orientation).lower()\n    if orientation == \"right\":\n        return img.rotate(-90, expand=True)\n    if orientation == \"left\":\n        return img.rotate(90, expand=True)\n    if orientation == \"bottom\":\n        return img.rotate(180, expand=True)\n    return img\n\n\ndef pad_square_resize(img: Image.Image, size: int, fill: np.ndarray = PAD) -> Image.Image:\n    rgb = np.asarray(img.convert(\"RGB\"))\n    h, w = rgb.shape[:2]\n    side = max(h, w)\n    canvas = np.zeros((side, side, 3), dtype=np.uint8)\n    canvas[:] = fill[None, None, :]\n    y0 = (side - h) // 2\n    x0 = (side - w) // 2\n    canvas[y0 : y0 + h, x0 : x0 + w] = rgb\n    return Image.fromarray(canvas).resize((size, size), Image.Resampling.LANCZOS)\n\n\ndef load_salamander_view(row: pd.Series, img_size: int, view: str) -> Image.Image:\n    img = Image.open(row[\"abs_path\"]).convert(\"RGB\")\n    if view == \"norm\":\n        img = orientation_normalize(img, str(row.get(\"orientation\", \"\")))\n    return pad_square_resize(img, img_size, PAD)\n\n\ndef make_five_crops(img: Image.Image) -> list[Image.Image]:\n    img = img.convert(\"RGB\")\n    side = min(img.size)\n    crop = max(16, int(round(side * 0.86)))\n    coords = [\n        (0, 0),\n        (side - crop, 0),\n        (0, side - crop),\n        (side - crop, side - crop),\n        ((side - crop) // 2, (side - crop) // 2),\n    ]\n    return [img.crop((x, y, x + crop, y + crop)) for x, y in coords]\n\n\ndef normalize_rows(arr: np.ndarray) -> np.ndarray:\n    arr = np.asarray(arr, dtype=np.float32)\n    norm = np.linalg.norm(arr, axis=1, keepdims=True)\n    return arr / np.maximum(norm, 1e-7)\n\n\nclass SimpleGlobalExtractor:\n    def __init__(self, img_size: int, view: str):\n        self.img_size = img_size\n        self.view = view\n        self.name = f\"simple_color_fallback:{view}\"\n\n    def extract(self, rows: pd.DataFrame) -> np.ndarray:\n        feats = []\n        for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=f\"simple {self.view}\"):\n            img = load_salamander_view(pd.Series(row._asdict()), self.img_size, self.view)\n            arr = np.asarray(img.resize((64, 64), Image.Resampling.BILINEAR), dtype=np.float32) / 255.0\n            hsv = cv2.cvtColor((arr * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)\n            hist_h = cv2.calcHist([hsv], [0], None, [32], [0, 180]).reshape(-1)\n            hist_s = cv2.calcHist([hsv], [1], None, [16], [0, 256]).reshape(-1)\n            hist_v = cv2.calcHist([hsv], [2], None, [16], [0, 256]).reshape(-1)\n            small = cv2.resize(arr, (8, 8)).reshape(-1)\n            feat = np.concatenate([hist_h, hist_s, hist_v, small]).astype(np.float32)\n            feat /= max(float(np.linalg.norm(feat)), 1e-7)\n            feats.append(feat)\n        return normalize_rows(np.vstack(feats))\n\n\nclass EVAGlobalExtractor:\n    def __init__(self, model_name: str, pretrained: str, batch_size: int, img_size: int, view: str):\n        ok = install_if_missing(\"open_clip\", \"open_clip_torch\")\n        if not ok:\n            raise ImportError(\"open_clip_torch unavailable\")\n        import torch\n        import open_clip\n\n        self.torch = torch\n        self.device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        self.batch_size = batch_size\n        self.img_size = img_size\n        self.view = view\n        candidates = [(model_name, pretrained), (\"ViT-L-14\", \"laion2b_s32b_b82k\"), (\"ViT-B-16\", \"laion2b_s34b_b88k\")]\n        last_exc = None\n        for name, weights in candidates:\n            try:\n                print(f\"[EVA/CLIP] loading {name} / {weights} for {view}\")\n                model, _, preprocess = open_clip.create_model_and_transforms(name, pretrained=weights)\n                self.model = model.to(self.device).eval()\n                self.preprocess = preprocess\n                self.name = f\"{name}:{weights}:{view}\"\n                return\n            except Exception as exc:\n                print(f\"[EVA/CLIP] failed {name}/{weights}: {exc}\")\n                last_exc = exc\n        raise RuntimeError(f\"Could not load EVA/CLIP model: {last_exc}\")\n\n    def extract(self, rows: pd.DataFrame) -> np.ndarray:\n        torch = self.torch\n        crop_feats = []\n        batch = []\n        with torch.no_grad():\n            for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=self.name):\n                img = load_salamander_view(pd.Series(row._asdict()), self.img_size, self.view)\n                for crop in make_five_crops(img):\n                    batch.append(self.preprocess(crop))\n                if len(batch) >= self.batch_size:\n                    crop_feats.extend(self._flush(batch))\n                    batch.clear()\n            if batch:\n                crop_feats.extend(self._flush(batch))\n        arr = np.vstack(crop_feats).astype(np.float32)\n        arr = arr.reshape(len(rows), 5, -1).mean(axis=1)\n        return normalize_rows(arr)\n\n    def _flush(self, batch: list) -> list[np.ndarray]:\n        torch = self.torch\n        x = torch.stack(batch).to(self.device)\n        feat = self.model.encode_image(x)\n        feat = feat / feat.norm(dim=-1, keepdim=True).clamp_min(1e-7)\n        return [v for v in feat.detach().cpu().float().numpy()]\n\n\n@dataclass\nclass LocalFeatures:\n    keypoints: np.ndarray\n    descriptors: np.ndarray\n\n\nclass ALIKEDExtractor:\n    def __init__(self, img_size: int, max_keypoints: int, view: str):\n        import torch\n\n        self.torch = torch\n        self.device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        self.img_size = img_size\n        self.max_keypoints = max_keypoints\n        self.view = view\n        self.backend = \"\"\n\n        try:\n            import kornia.feature as KF\n\n            if hasattr(KF, \"ALIKED\"):\n                try:\n                    self.model = KF.ALIKED.from_pretrained(\n                        \"aliked-n16\",\n                        max_num_keypoints=max_keypoints,\n                        detection_threshold=0.18,\n                    ).to(self.device).eval()\n                except TypeError:\n                    self.model = KF.ALIKED.from_pretrained(\"aliked-n16\").to(self.device).eval()\n                self.backend = \"kornia\"\n                print(f\"[ALIKED] backend=kornia view={view} device={self.device}\")\n                return\n            print(\"[ALIKED] kornia has no ALIKED; trying LightGlue\")\n        except Exception as exc:\n            print(f\"[ALIKED] kornia unavailable: {exc}; trying LightGlue\")\n\n        try:\n            try:\n                from lightglue import ALIKED as LG_ALIKED\n            except Exception:\n                try:\n                    print(\"[deps] installing lightglue\")\n                    subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"lightglue\"], check=True)\n                except Exception:\n                    print(\"[deps] pip lightglue failed; trying GitHub\")\n                    subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"git+https://github.com/cvg/LightGlue.git\"], check=True)\n                from lightglue import ALIKED as LG_ALIKED\n            self.model = LG_ALIKED(max_num_keypoints=max_keypoints, detection_threshold=0.18).to(self.device).eval()\n            self.backend = \"lightglue\"\n            print(f\"[ALIKED] backend=lightglue view={view} device={self.device}\")\n        except Exception as exc:\n            raise ImportError(f\"Could not initialize ALIKED: {exc}\") from exc\n\n    def extract(self, rows: pd.DataFrame) -> dict[int, LocalFeatures]:\n        out = {}\n        torch = self.torch\n        with torch.no_grad():\n            for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=f\"ALIKED {self.view}\"):\n                img = load_salamander_view(pd.Series(row._asdict()), self.img_size, self.view)\n                arr = np.asarray(img, dtype=np.float32) / 255.0\n                tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(self.device)\n                try:\n                    raw = self.model.extract(tensor) if self.backend == \"lightglue\" else self.model(tensor)\n                    features = raw[0] if isinstance(raw, (list, tuple)) else raw\n                    kpts = feature_value(features, [\"keypoints\", \"kpts\"])\n                    desc = feature_value(features, [\"descriptors\", \"desc\"])\n                    kpts_np = squeeze_keypoints(tensor_to_numpy(kpts))\n                    desc_np = squeeze_descriptors(tensor_to_numpy(desc), len(kpts_np))\n                except Exception as exc:\n                    print(f\"[ALIKED] failed image_id={row.image_id} view={self.view}: {exc}\")\n                    kpts_np = np.zeros((0, 2), dtype=np.float32)\n                    desc_np = np.zeros((0, 128), dtype=np.float32)\n                if len(kpts_np) > self.max_keypoints:\n                    kpts_np = kpts_np[: self.max_keypoints]\n                    desc_np = desc_np[: self.max_keypoints]\n                out[int(row.image_id)] = LocalFeatures(kpts_np, desc_np)\n        return out\n\n\ndef tensor_to_numpy(value) -> np.ndarray:\n    if value is None:\n        return np.zeros((0,), dtype=np.float32)\n    if hasattr(value, \"detach\"):\n        return value.detach().cpu().float().numpy()\n    return np.asarray(value, dtype=np.float32)\n\n\ndef squeeze_keypoints(arr: np.ndarray) -> np.ndarray:\n    arr = np.asarray(arr, dtype=np.float32)\n    if arr.size == 0:\n        return np.zeros((0, 2), dtype=np.float32)\n    while arr.ndim > 2 and arr.shape[0] == 1:\n        arr = arr[0]\n    arr = arr.reshape(-1, arr.shape[-1])\n    if arr.shape[1] > 2:\n        arr = arr[:, :2]\n    if arr.shape[1] < 2:\n        return np.zeros((0, 2), dtype=np.float32)\n    return arr.astype(np.float32, copy=False)\n\n\ndef squeeze_descriptors(arr: np.ndarray, n_keypoints: int) -> np.ndarray:\n    arr = np.asarray(arr, dtype=np.float32)\n    if arr.size == 0 or n_keypoints <= 0:\n        return np.zeros((0, 128), dtype=np.float32)\n    while arr.ndim > 2 and arr.shape[0] == 1:\n        arr = arr[0]\n    if arr.ndim == 1:\n        arr = arr.reshape(1, -1)\n    if arr.shape[0] != n_keypoints and arr.ndim == 2 and arr.shape[1] == n_keypoints:\n        arr = arr.T\n    arr = arr.reshape(arr.shape[0], -1)\n    if arr.shape[0] != n_keypoints:\n        arr = arr[: min(arr.shape[0], n_keypoints)]\n    return arr.astype(np.float32, copy=False)\n\n\ndef feature_value(features, names: list[str]):\n    if isinstance(features, dict):\n        for name in names:\n            if name in features:\n                return features[name]\n    for name in names:\n        if hasattr(features, name):\n            return getattr(features, name)\n    return None\n\n\ndef match_local(fa: LocalFeatures, fb: LocalFeatures, sim_threshold: float) -> dict[str, float]:\n    if fa.descriptors.size == 0 or fb.descriptors.size == 0 or len(fa.keypoints) < 4 or len(fb.keypoints) < 4:\n        return {\"local_score\": 0.0, \"matches\": 0, \"inliers\": 0, \"mean_conf\": 0.0}\n    da = normalize_rows(fa.descriptors)\n    db = normalize_rows(fb.descriptors)\n    sim = da @ db.T\n    nn_ab = sim.argmax(axis=1)\n    conf_ab = sim[np.arange(sim.shape[0]), nn_ab]\n    nn_ba = sim.argmax(axis=0)\n    mutual = np.array([nn_ba[j] == i for i, j in enumerate(nn_ab)], dtype=bool)\n    keep = mutual & (conf_ab >= sim_threshold)\n    idx_a = np.where(keep)[0]\n    idx_b = nn_ab[idx_a]\n    matches = int(len(idx_a))\n    mean_conf = float(conf_ab[idx_a].mean()) if matches else 0.0\n    inliers = 0\n    if matches >= 4:\n        pts_a = fa.keypoints[idx_a].astype(np.float32)\n        pts_b = fb.keypoints[idx_b].astype(np.float32)\n        try:\n            _, mask_h = cv2.findHomography(pts_a, pts_b, cv2.RANSAC, 12.0, maxIters=1000)\n            inliers_h = int(mask_h.sum()) if mask_h is not None else 0\n        except Exception:\n            inliers_h = 0\n        try:\n            _, mask_a = cv2.estimateAffinePartial2D(pts_a, pts_b, method=cv2.RANSAC, ransacReprojThreshold=12.0)\n            inliers_a = int(mask_a.sum()) if mask_a is not None else 0\n        except Exception:\n            inliers_a = 0\n        inliers = max(inliers_h, inliers_a)\n    local_score = 0.58 * min(inliers / 16.0, 1.0) + 0.27 * min(matches / 42.0, 1.0) + 0.15 * np.clip(mean_conf, 0, 1)\n    return {\"local_score\": float(local_score), \"matches\": matches, \"inliers\": int(inliers), \"mean_conf\": float(mean_conf)}\n\n\ndef orb_pair_score(row_a: pd.Series, row_b: pd.Series, img_size: int, view: str) -> dict[str, float]:\n    img_a = np.asarray(load_salamander_view(row_a, img_size, view).convert(\"L\"))\n    img_b = np.asarray(load_salamander_view(row_b, img_size, view).convert(\"L\"))\n    orb = cv2.ORB_create(nfeatures=800, fastThreshold=7)\n    kpa, desa = orb.detectAndCompute(img_a, None)\n    kpb, desb = orb.detectAndCompute(img_b, None)\n    if desa is None or desb is None or not kpa or not kpb:\n        return {\"local_score\": 0.0, \"matches\": 0, \"inliers\": 0, \"mean_conf\": 0.0}\n    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)\n    matches = sorted(bf.match(desa, desb), key=lambda m: m.distance)[:80]\n    mean_conf = float(np.mean([1.0 - min(m.distance, 96) / 96.0 for m in matches])) if matches else 0.0\n    inliers = 0\n    if len(matches) >= 4:\n        pts_a = np.float32([kpa[m.queryIdx].pt for m in matches])\n        pts_b = np.float32([kpb[m.trainIdx].pt for m in matches])\n        try:\n            _, mask = cv2.estimateAffinePartial2D(pts_a, pts_b, method=cv2.RANSAC, ransacReprojThreshold=12.0)\n            inliers = int(mask.sum()) if mask is not None else 0\n        except Exception:\n            inliers = 0\n    local_score = 0.58 * min(inliers / 16.0, 1.0) + 0.27 * min(len(matches) / 42.0, 1.0) + 0.15 * mean_conf\n    return {\"local_score\": float(local_score), \"matches\": int(len(matches)), \"inliers\": int(inliers), \"mean_conf\": mean_conf}\n\n\ndef load_feature_cache(path: Path, n_rows: int) -> np.ndarray | None:\n    if path.exists():\n        arr = np.load(path)\n        if arr.shape[0] == n_rows:\n            print(f\"[cache] loaded {path} {arr.shape}\")\n            return normalize_rows(arr)\n        print(f\"[cache] ignoring wrong row count {path}: {arr.shape[0]} != {n_rows}\")\n    return None\n\n\ndef load_local_cache(path: Path) -> dict[int, LocalFeatures] | None:\n    if not path.exists():\n        return None\n    try:\n        with path.open(\"rb\") as f:\n            raw = pickle.load(f)\n        out = {\n            int(k): LocalFeatures(np.asarray(v[\"keypoints\"], dtype=np.float32), np.asarray(v[\"descriptors\"], dtype=np.float32))\n            for k, v in raw.items()\n        }\n        print(f\"[cache] loaded local features {path}\")\n        return out\n    except Exception as exc:\n        print(f\"[cache] could not load local features {path}: {exc}\")\n        return None\n\n\ndef save_local_cache(path: Path, features: dict[int, LocalFeatures]) -> None:\n    raw = {int(k): {\"keypoints\": v.keypoints.astype(np.float32), \"descriptors\": v.descriptors.astype(np.float32)} for k, v in features.items()}\n    with path.open(\"wb\") as f:\n        pickle.dump(raw, f, protocol=pickle.HIGHEST_PROTOCOL)\n    print(f\"[cache] wrote local features {path}\")\n\n\ndef extract_global_views(args: argparse.Namespace, rows: pd.DataFrame, reports_dir: Path) -> dict[str, np.ndarray]:\n    out = {}\n    for view in [\"orig\", \"norm\"]:\n        path = reports_dir / f\"{VERSION}_global_{view}.npy\"\n        arr = load_feature_cache(path, len(rows))\n        if arr is None:\n            extractor = SimpleGlobalExtractor(args.img_size, view) if args.disable_eva else EVAGlobalExtractor(args.eva_model, args.eva_pretrained, args.eva_batch_size, args.img_size, view)\n            arr = extractor.extract(rows)\n            np.save(path, arr)\n            print(f\"[features] wrote {path} {arr.shape}\")\n        out[view] = arr\n    return out\n\n\ndef extract_local_views(args: argparse.Namespace, rows: pd.DataFrame, reports_dir: Path) -> dict[str, dict[int, LocalFeatures]]:\n    out = {}\n    if args.disable_aliked:\n        return out\n    for view in [\"orig\", \"norm\"]:\n        path = reports_dir / f\"{VERSION}_aliked_{view}.pkl\"\n        cached = load_local_cache(path)\n        if cached is not None:\n            out[view] = cached\n            continue\n        extractor = ALIKEDExtractor(args.img_size, args.max_aliked_keypoints, view)\n        feats = extractor.extract(rows)\n        save_local_cache(path, feats)\n        out[view] = feats\n    return out\n\n\ndef sample_train_pairs(train: pd.DataFrame, features: np.ndarray, limit: int) -> list[tuple[int, int, int]]:\n    ids = train[\"image_id\"].astype(int).to_numpy()\n    labels = train[\"identity\"].astype(str).to_numpy()\n    by_label = {}\n    for idx, label in enumerate(labels):\n        by_label.setdefault(label, []).append(idx)\n    sim = features @ features.T\n    rng = np.random.default_rng(SEED)\n    pairs: set[tuple[int, int, int]] = set()\n    for idxs in by_label.values():\n        if len(idxs) < 2:\n            continue\n        combos = list(itertools.combinations(idxs, 2))\n        if len(combos) > 50:\n            hard = sorted(combos, key=lambda ab: sim[ab[0], ab[1]])[:20]\n            rand = [combos[i] for i in rng.choice(len(combos), size=30, replace=False)]\n            combos = hard + rand\n        for a, b in combos:\n            x, y = sorted((int(ids[a]), int(ids[b])))\n            pairs.add((x, y, 1))\n    order = np.argsort(-sim, axis=1)\n    for i in range(len(ids)):\n        added = 0\n        for j in order[i, 1:80]:\n            if labels[i] != labels[j]:\n                x, y = sorted((int(ids[i]), int(ids[j])))\n                pairs.add((x, y, 0))\n                added += 1\n                if added >= 10:\n                    break\n    while len([p for p in pairs if p[2] == 0]) < min(limit // 2, 5000):\n        a, b = rng.integers(0, len(ids), size=2)\n        if a == b or labels[a] == labels[b]:\n            continue\n        x, y = sorted((int(ids[a]), int(ids[b])))\n        pairs.add((x, y, 0))\n    positives = [p for p in pairs if p[2] == 1]\n    negatives = [p for p in pairs if p[2] == 0]\n    rng.shuffle(positives)\n    rng.shuffle(negatives)\n    selected = positives[: limit // 2] + negatives[: limit - min(len(positives), limit // 2)]\n    rng.shuffle(selected)\n    return selected[:limit]\n\n\ndef test_candidate_pairs(test: pd.DataFrame, features: np.ndarray, base: pd.DataFrame, top_k: int, limit: int) -> list[tuple[int, int, str]]:\n    ids = test[\"image_id\"].astype(int).to_numpy()\n    cluster_by_id = base.set_index(\"image_id\")[\"cluster\"].astype(str).to_dict()\n    sim = features @ features.T\n    order = np.argsort(-sim, axis=1)\n    pairs: set[tuple[int, int, str]] = set()\n    for i in range(len(ids)):\n        added = 0\n        for j in order[i, 1 : top_k * 5]:\n            if cluster_by_id[int(ids[i])] == cluster_by_id[int(ids[j])]:\n                continue\n            x, y = sorted((int(ids[i]), int(ids[j])))\n            pairs.add((x, y, \"topk_cross\"))\n            added += 1\n            if added >= top_k:\n                break\n    id_to_idx = {int(image_id): i for i, image_id in enumerate(ids)}\n    return sorted(pairs, key=lambda p: -sim[id_to_idx[p[0]], id_to_idx[p[1]]])[:limit]\n\n\ndef score_pairs(\n    pairs: list[tuple[int, int, int | str]],\n    rows: pd.DataFrame,\n    feature_rows: pd.DataFrame,\n    global_features: dict[str, np.ndarray],\n    local_features: dict[str, dict[int, LocalFeatures]],\n    args: argparse.Namespace,\n) -> pd.DataFrame:\n    rows_by_id = rows.set_index(\"image_id\", drop=False)\n    feat_idx = {int(v): i for i, v in enumerate(feature_rows[\"image_id\"].astype(int).tolist())}\n    records = []\n    for a, b, tag in tqdm(pairs, desc=\"pair scores\"):\n        a = int(a)\n        b = int(b)\n        ia = feat_idx[a]\n        ib = feat_idx[b]\n        eva_orig = float(np.dot(global_features[\"orig\"][ia], global_features[\"orig\"][ib]))\n        eva_norm = float(np.dot(global_features[\"norm\"][ia], global_features[\"norm\"][ib]))\n        eva_best = max(eva_orig, eva_norm)\n        best_view = \"orig\" if eva_orig >= eva_norm else \"norm\"\n\n        local_records = {}\n        for view in [\"orig\", \"norm\"]:\n            if view in local_features:\n                local_records[view] = match_local(local_features[view][a], local_features[view][b], args.aliked_match_threshold)\n            elif args.allow_orb_fallback:\n                local_records[view] = orb_pair_score(rows_by_id.loc[a], rows_by_id.loc[b], args.img_size, view)\n            else:\n                local_records[view] = {\"local_score\": 0.0, \"matches\": 0, \"inliers\": 0, \"mean_conf\": 0.0}\n        local_best_view = \"orig\" if local_records[\"orig\"][\"local_score\"] >= local_records[\"norm\"][\"local_score\"] else \"norm\"\n        local_best = local_records[local_best_view]\n        rec = {\n            \"image_id_a\": a,\n            \"image_id_b\": b,\n            \"pair_tag\": tag,\n            \"eva_orig\": eva_orig,\n            \"eva_norm\": eva_norm,\n            \"eva_best\": eva_best,\n            \"eva_best01\": float((eva_best + 1.0) * 0.5),\n            \"eva_best_view\": best_view,\n            \"local_orig\": local_records[\"orig\"][\"local_score\"],\n            \"local_norm\": local_records[\"norm\"][\"local_score\"],\n            \"local_best\": local_best[\"local_score\"],\n            \"local_best_view\": local_best_view,\n            \"matches\": int(local_best[\"matches\"]),\n            \"inliers\": int(local_best[\"inliers\"]),\n            \"mean_conf\": float(local_best[\"mean_conf\"]),\n            \"orientation_a\": str(rows_by_id.loc[a].get(\"orientation\", \"\")),\n            \"orientation_b\": str(rows_by_id.loc[b].get(\"orientation\", \"\")),\n        }\n        if isinstance(tag, (int, np.integer)):\n            rec[\"label\"] = int(tag)\n        records.append(rec)\n    return pd.DataFrame(records)\n\n\ndef classifier_features(df: pd.DataFrame) -> np.ndarray:\n    arr = np.column_stack(\n        [\n            df[\"eva_best01\"].astype(float).to_numpy(),\n            ((df[\"eva_orig\"].astype(float).to_numpy() + 1.0) * 0.5),\n            ((df[\"eva_norm\"].astype(float).to_numpy() + 1.0) * 0.5),\n            df[\"local_best\"].astype(float).to_numpy(),\n            df[\"local_orig\"].astype(float).to_numpy(),\n            df[\"local_norm\"].astype(float).to_numpy(),\n            np.minimum(df[\"matches\"].astype(float).to_numpy() / 60.0, 1.0),\n            np.minimum(df[\"inliers\"].astype(float).to_numpy() / 24.0, 1.0),\n            df[\"mean_conf\"].astype(float).to_numpy(),\n            df[\"eva_best_view\"].astype(str).eq(\"norm\").astype(float).to_numpy(),\n            df[\"local_best_view\"].astype(str).eq(\"norm\").astype(float).to_numpy(),\n            df[\"orientation_a\"].astype(str).eq(df[\"orientation_b\"].astype(str)).astype(float).to_numpy(),\n        ]\n    )\n    return np.nan_to_num(arr.astype(np.float32), nan=0.0, posinf=1.0, neginf=0.0)\n\n\ndef train_calibrator(train_pairs: pd.DataFrame):\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.metrics import average_precision_score, roc_auc_score\n    from sklearn.model_selection import train_test_split\n\n    x = classifier_features(train_pairs)\n    y = train_pairs[\"label\"].astype(int).to_numpy()\n    x_tr, x_va, y_tr, y_va = train_test_split(x, y, test_size=0.25, random_state=SEED, stratify=y)\n    try:\n        clf = HistGradientBoostingClassifier(\n            max_iter=180,\n            learning_rate=0.045,\n            max_leaf_nodes=15,\n            l2_regularization=0.08,\n            random_state=SEED,\n        )\n        clf.fit(x_tr, y_tr)\n    except Exception as exc:\n        print(f\"[calibration] HGB failed; using logistic regression: {exc}\")\n        clf = LogisticRegression(max_iter=1000, class_weight=\"balanced\", random_state=SEED)\n        clf.fit(x_tr, y_tr)\n    pred = clf.predict_proba(x_va)[:, 1]\n    stats = {\n        \"train_pairs\": int(len(y)),\n        \"positive_pairs\": int(y.sum()),\n        \"negative_pairs\": int((y == 0).sum()),\n        \"valid_auc\": float(roc_auc_score(y_va, pred)) if len(np.unique(y_va)) > 1 else 0.5,\n        \"valid_ap\": float(average_precision_score(y_va, pred)) if len(np.unique(y_va)) > 1 else float(y_va.mean()),\n    }\n    return clf, stats\n\n\ndef select_rule(train_pairs: pd.DataFrame, target_precision: float, min_support: int) -> dict:\n    candidates = []\n    for prob_thr in [0.90, 0.94, 0.97, 0.98, 0.99]:\n        for local_thr in [0.40, 0.55, 0.70, 0.82, 0.90, 0.95]:\n            for inlier_thr in [4, 6, 8, 10, 14, 18, 24, 32]:\n                for eva_thr in [0.86, 0.90, 0.94, 0.96, 0.98]:\n                    mask = (\n                        (train_pairs[\"match_prob\"].astype(float) >= prob_thr)\n                        & (train_pairs[\"local_best\"].astype(float) >= local_thr)\n                        & (train_pairs[\"inliers\"].astype(float) >= inlier_thr)\n                        & (train_pairs[\"eva_best01\"].astype(float) >= eva_thr)\n                    )\n                    sub = train_pairs[mask]\n                    if len(sub) < min_support:\n                        continue\n                    precision = float(sub[\"label\"].mean())\n                    positives = int(sub[\"label\"].sum())\n                    negatives = int((sub[\"label\"] == 0).sum())\n                    candidates.append(\n                        {\n                            \"prob_thr\": prob_thr,\n                            \"local_thr\": local_thr,\n                            \"inlier_thr\": inlier_thr,\n                            \"eva_thr\": eva_thr,\n                            \"support\": int(len(sub)),\n                            \"positives\": positives,\n                            \"negatives\": negatives,\n                            \"precision\": precision,\n                        }\n                    )\n    if not candidates:\n        return {\"prob_thr\": 0.999, \"local_thr\": 0.999, \"inlier_thr\": 999, \"eva_thr\": 0.999, \"support\": 0, \"positives\": 0, \"negatives\": 0, \"precision\": 0.0}\n    df = pd.DataFrame(candidates)\n    good = df[df[\"precision\"] >= target_precision].copy()\n    if good.empty:\n        good = df.sort_values([\"precision\", \"positives\", \"support\"], ascending=[False, False, False]).head(30)\n    else:\n        good = good.sort_values([\"positives\", \"precision\", \"support\"], ascending=[False, False, False])\n    return good.iloc[0].to_dict()\n\n\ndef build_variant(\n    name: str,\n    base: pd.DataFrame,\n    sample: pd.DataFrame,\n    test_rows: pd.DataFrame,\n    salamander_test: pd.DataFrame,\n    test_pairs: pd.DataFrame,\n    rule: dict,\n    max_size: int,\n    max_edges: int,\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    salamander_ids = salamander_test[\"image_id\"].astype(int).tolist()\n    current = base[base[\"image_id\"].astype(int).isin(salamander_ids)].copy()\n    uf = UF(salamander_ids)\n    for _, group in current.groupby(\"cluster\"):\n        ids = group[\"image_id\"].astype(int).tolist()\n        for a, b in itertools.combinations(ids, 2):\n            uf.union(a, b)\n\n    cluster_by_id = current.set_index(\"image_id\")[\"cluster\"].astype(str).to_dict()\n    candidates = test_pairs[\n        (test_pairs[\"match_prob\"].astype(float) >= float(rule[\"prob_thr\"]))\n        & (test_pairs[\"local_best\"].astype(float) >= float(rule[\"local_thr\"]))\n        & (test_pairs[\"inliers\"].astype(float) >= int(rule[\"inlier_thr\"]))\n        & (test_pairs[\"eva_best01\"].astype(float) >= float(rule[\"eva_thr\"]))\n    ].copy()\n    candidates = candidates.sort_values([\"match_prob\", \"local_best\", \"inliers\"], ascending=False)\n\n    actions = []\n    for row in candidates.itertuples(index=False):\n        if len(actions) >= max_edges:\n            break\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        if a not in uf.parent or b not in uf.parent or uf.find(a) == uf.find(b):\n            continue\n        size_after = uf.size[uf.find(a)] + uf.size[uf.find(b)]\n        if size_after > max_size:\n            continue\n        uf.union(a, b)\n        actions.append(\n            {\n                \"profile\": name,\n                \"image_id_a\": a,\n                \"image_id_b\": b,\n                \"cluster_a\": cluster_by_id[a],\n                \"cluster_b\": cluster_by_id[b],\n                \"match_prob\": float(row.match_prob),\n                \"eva_best01\": float(row.eva_best01),\n                \"eva_best_view\": row.eva_best_view,\n                \"local_best\": float(row.local_best),\n                \"local_best_view\": row.local_best_view,\n                \"matches\": int(row.matches),\n                \"inliers\": int(row.inliers),\n                \"size_after\": int(size_after),\n            }\n        )\n\n    groups = {}\n    for image_id in salamander_ids:\n        groups.setdefault(uf.find(image_id), []).append(image_id)\n    labels = {}\n    for idx, members in enumerate(sorted(groups.values(), key=lambda xs: (min(xs), len(xs), max(xs)))):\n        for image_id in members:\n            labels[image_id] = f\"cluster_{SALAMANDER}_{idx}\"\n\n    sub = base.copy()\n    mask = sub[\"image_id\"].astype(int).isin(labels)\n    sub.loc[mask, \"cluster\"] = sub.loc[mask, \"image_id\"].astype(int).map(labels)\n    final = compact_species_labels(sub, sample, test_rows)\n    shape = shape_report(final, test_rows)\n    return final, pd.DataFrame(actions), shape\n\n\ndef shape_report(sub: pd.DataFrame, test_rows: pd.DataFrame) -> pd.DataFrame:\n    tmp = sub.merge(test_rows[[\"image_id\", \"dataset\"]], on=\"image_id\", how=\"left\")\n    rows = []\n    for species, group in tmp.groupby(\"dataset\", sort=True):\n        sizes = group.groupby(\"cluster\").size()\n        rows.append({\"species\": species, \"n_images\": int(len(group)), \"n_clusters\": int(sizes.size), \"max_cluster_size\": int(sizes.max()), \"singletons\": int((sizes == 1).sum())})\n    return pd.DataFrame(rows)\n\n\ndef save_visualizations(out_dir: Path, salamander_test: pd.DataFrame, actions: pd.DataFrame, img_size: int, max_pairs: int = 16) -> None:\n    if actions.empty:\n        return\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rows_by_id = salamander_test.set_index(\"image_id\", drop=False)\n    pairs = actions.head(max_pairs)\n    for idx, row in enumerate(pairs.itertuples(index=False)):\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        panels = []\n        for image_id in [a, b]:\n            r = rows_by_id.loc[image_id]\n            original = ImageOps.contain(Image.open(r.abs_path).convert(\"RGB\"), (img_size, img_size))\n            norm = load_salamander_view(r, img_size, \"norm\")\n            panels.extend([original, norm])\n        canvas = Image.new(\"RGB\", (img_size * 4, img_size + 44), (238, 238, 238))\n        for j, img in enumerate(panels):\n            bg = Image.new(\"RGB\", (img_size, img_size), tuple(int(x) for x in PAD))\n            bg.paste(img, ((img_size - img.width) // 2, (img_size - img.height) // 2))\n            canvas.paste(bg, (j * img_size, 0))\n        draw = ImageDraw.Draw(canvas)\n        text = f\"{a} vs {b} prob={row.match_prob:.3f} local={row.local_best:.3f} inliers={row.inliers} view={row.local_best_view}\"\n        draw.rectangle((0, img_size, img_size * 4, img_size + 44), fill=(245, 245, 245))\n        draw.text((10, img_size + 14), text, fill=(20, 20, 20))\n        canvas.save(out_dir / f\"salamander_merge_{idx:02d}_{a}_{b}.jpg\", quality=92)\n\n\ndef main() -> None:\n    args = parse_args()\n    seed_everything()\n    if args.smoke:\n        args.disable_eva = True\n        args.disable_aliked = True\n        args.train_pair_limit = min(args.train_pair_limit, 500)\n        args.test_pair_limit = min(args.test_pair_limit, 800)\n        args.top_k = min(args.top_k, 8)\n\n    output_root = Path(args.output_root)\n    reports_dir = output_root / \"reports\"\n    submissions_dir = output_root / \"submissions\"\n    viz_dir = output_root / \"visualizations\"\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    submissions_dir.mkdir(parents=True, exist_ok=True)\n\n    data_root = find_data_root(args.data_root)\n    metadata = prepare_metadata(data_root)\n    test_rows = metadata[metadata[\"split\"].eq(\"test\")].copy()\n    sample = pd.read_csv(data_root / \"sample_submission.csv\")[[\"image_id\"]].copy()\n    sample[\"image_id\"] = sample[\"image_id\"].astype(int)\n\n    current_best_path = find_optional_file(CURRENT_BEST_NAMES, args.current_best, rank_current_best)\n    if (\n        current_best_path is not None\n        and args.current_best is None\n        and current_best_path.name == \"submission.csv\"\n        and rank_current_best(current_best_path)[0] >= 900\n    ):\n        current_best_path = None\n    if current_best_path is not None:\n        current_best = sample.merge(load_submission(current_best_path), on=\"image_id\", how=\"left\")\n        validate_submission(current_best, sample)\n        base_source = str(current_best_path)\n    else:\n        base_path = find_file(BASE_032684_NAMES, args.base_032684, rank_base_032684)\n        lynx_pair_path = find_file([LYNX_PAIR_SCORE_NAME], args.lynx_pair_scores, rank_pair_path)\n        base_032684 = sample.merge(load_submission(base_path), on=\"image_id\", how=\"left\")\n        validate_submission(base_032684, sample)\n        current_best = build_032903_from_components(base_032684, pd.read_csv(lynx_pair_path), sample, test_rows)\n        validate_submission(current_best, sample)\n        base_source = f\"rebuilt_032903_from base={base_path} lynx_pairs={lynx_pair_path}\"\n\n    salamander_train = metadata[(metadata[\"dataset\"].eq(SALAMANDER)) & (metadata[\"split\"].eq(\"train\"))].copy()\n    salamander_test = metadata[(metadata[\"dataset\"].eq(SALAMANDER)) & (metadata[\"split\"].eq(\"test\"))].copy()\n    if args.smoke:\n        salamander_train = salamander_train.head(260).copy()\n        salamander_test = salamander_test.head(120).copy()\n    all_salamander = pd.concat([salamander_train, salamander_test], ignore_index=True)\n\n    print(f\"[paths] data_root={data_root}\")\n    print(f\"[paths] current_best_source={base_source}\")\n    print(f\"[data] salamander_train={len(salamander_train)} salamander_test={len(salamander_test)}\")\n\n    t0 = time.time()\n    global_features = extract_global_views(args, all_salamander, reports_dir)\n    local_features = extract_local_views(args, all_salamander, reports_dir)\n\n    combined_for_sampling = normalize_rows(global_features[\"orig\"] + global_features[\"norm\"])\n    train_features = combined_for_sampling[: len(salamander_train)]\n    test_features = combined_for_sampling[len(salamander_train) :]\n\n    train_pair_idx = sample_train_pairs(salamander_train, train_features, args.train_pair_limit)\n    train_pairs = score_pairs(train_pair_idx, all_salamander, all_salamander, global_features, local_features, args)\n    train_pairs.to_csv(reports_dir / f\"{VERSION}_train_pairs.csv\", index=False)\n    clf, cal_stats = train_calibrator(train_pairs)\n    train_pairs[\"match_prob\"] = clf.predict_proba(classifier_features(train_pairs))[:, 1]\n    train_pairs.to_csv(reports_dir / f\"{VERSION}_train_pairs_scored.csv\", index=False)\n    print(f\"[calibration] {json.dumps(cal_stats, indent=2)}\")\n\n    test_pair_idx = test_candidate_pairs(salamander_test, test_features, current_best, args.top_k, args.test_pair_limit)\n    test_pairs = score_pairs(test_pair_idx, all_salamander, all_salamander, global_features, local_features, args)\n    cluster_map = current_best.set_index(\"image_id\")[\"cluster\"].astype(str).to_dict()\n    test_pairs[\"cluster_a\"] = test_pairs[\"image_id_a\"].astype(int).map(cluster_map)\n    test_pairs[\"cluster_b\"] = test_pairs[\"image_id_b\"].astype(int).map(cluster_map)\n    test_pairs[\"same_current_cluster\"] = test_pairs[\"cluster_a\"].eq(test_pairs[\"cluster_b\"])\n    test_pairs[\"match_prob\"] = clf.predict_proba(classifier_features(test_pairs))[:, 1]\n    test_pairs = test_pairs.sort_values([\"match_prob\", \"local_best\", \"inliers\"], ascending=False).reset_index(drop=True)\n    test_pairs.to_csv(reports_dir / f\"{VERSION}_test_pair_scores.csv\", index=False)\n\n    rules = {\n        \"ultra_p90\": select_rule(train_pairs, target_precision=0.995, min_support=8),\n        \"strict_p90\": select_rule(train_pairs, target_precision=0.985, min_support=14),\n        \"p95_gamble\": select_rule(train_pairs, target_precision=0.975, min_support=20),\n    }\n    for key, rule in rules.items():\n        rule[\"profile\"] = key\n    pd.DataFrame(rules.values()).to_csv(reports_dir / f\"{VERSION}_selected_rules.csv\", index=False)\n\n    profiles = [\n        (\"ultra_p90\", 5, 10),\n        (\"strict_p90\", 5, 18),\n        (\"p95_gamble\", 7, 28),\n    ]\n    written = {}\n    action_frames = []\n    shape_frames = []\n    for profile, max_size, max_edges in profiles:\n        sub, actions, shape = build_variant(profile, current_best, sample, test_rows, salamander_test, test_pairs, rules[profile], max_size=max_size, max_edges=max_edges)\n        validate_submission(sub, sample)\n        out_name = f\"submission_{VERSION}_{profile}.csv\"\n        sub.to_csv(submissions_dir / out_name, index=False)\n        if profile == \"ultra_p90\":\n            sub.to_csv(output_root / \"submission.csv\", index=False)\n        actions.to_csv(reports_dir / f\"{VERSION}_{profile}_actions.csv\", index=False)\n        shape[\"profile\"] = profile\n        shape.to_csv(reports_dir / f\"{VERSION}_{profile}_shape_report.csv\", index=False)\n        actions[\"profile\"] = profile\n        action_frames.append(actions)\n        shape_frames.append(shape)\n        written[profile] = str(submissions_dir / out_name)\n        print(f\"\\n[{profile}] rule={rules[profile]}\")\n        print(\"actions:\", len(actions))\n        print(shape.to_string(index=False))\n        if args.save_visualizations and profile == \"ultra_p90\":\n            save_visualizations(viz_dir, salamander_test, actions, args.img_size)\n\n    all_actions = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()\n    all_shapes = pd.concat(shape_frames, ignore_index=True) if shape_frames else pd.DataFrame()\n    all_actions.to_csv(reports_dir / f\"{VERSION}_all_actions.csv\", index=False)\n    all_shapes.to_csv(reports_dir / f\"{VERSION}_all_shape_reports.csv\", index=False)\n    base_shape = shape_report(current_best, test_rows)\n    base_shape.to_csv(reports_dir / f\"{VERSION}_base_shape_report.csv\", index=False)\n\n    summary = {\n        \"version\": VERSION,\n        \"base_public_lb\": \"0.32903\",\n        \"data_root\": str(data_root),\n        \"current_best_source\": base_source,\n        \"calibration\": cal_stats,\n        \"selected_rules\": rules,\n        \"written\": written,\n        \"recommended_first\": str(output_root / \"submission.csv\"),\n        \"elapsed_minutes\": round((time.time() - t0) / 60.0, 2),\n        \"notes\": [\n            \"Salamander-only merge candidates; no splits.\",\n            \"Two views are scored for every image: original orientation and winner-style orientation-normalized.\",\n            \"Default submission.csv is ultra_p90. Inspect selected_rules/actions/shape reports before upload.\",\n        ],\n    }\n    (reports_dir / f\"{VERSION}_summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    print(\"\\n[done]\")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()\n"
Path("salamander_eva_aliked_mergeonly_v20260430.py").write_text(SCRIPT_TEXT, encoding="utf-8")
print("wrote salamander_eva_aliked_mergeonly_v20260430.py")
            

In [ ]:
import importlib.util
import subprocess
import sys
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

for import_name, pip_name in [
    ("cv2", "opencv-python-headless"),
    ("open_clip", "open_clip_torch"),
    ("kornia", "kornia"),
    ("lightglue", "lightglue"),
]:
    if importlib.util.find_spec(import_name) is None:
        print("installing", pip_name)
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
        except Exception:
            if import_name == "lightglue":
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/cvg/LightGlue.git"], check=True)
            else:
                raise
    else:
        print(import_name, "already available")
            

In [ ]:
from pathlib import Path
import subprocess
import sys

OUTPUT_ROOT = Path("/kaggle/working/animalclef_salamander_eva_aliked_mergeonly_v20260430")

# Usually keep these as None.
# If auto-detection picks the wrong input because many old notebooks are attached,
# fill the exact paths here.
CURRENT_BEST_CSV = None
BASE_032684_CSV = None
LYNX_PAIR_SCORES_CSV = None

cmd = [
    sys.executable,
    "salamander_eva_aliked_mergeonly_v20260430.py",
    "--output-root", str(OUTPUT_ROOT),
    "--img-size", "512",
    "--top-k", "18",
    "--train-pair-limit", "7000",
    "--test-pair-limit", "14000",
    "--max-aliked-keypoints", "512",
    "--eva-batch-size", "32",
    "--save-visualizations",
]
if CURRENT_BEST_CSV:
    cmd.extend(["--current-best", CURRENT_BEST_CSV])
if BASE_032684_CSV:
    cmd.extend(["--base-032684", BASE_032684_CSV])
if LYNX_PAIR_SCORES_CSV:
    cmd.extend(["--lynx-pair-scores", LYNX_PAIR_SCORES_CSV])

print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)
            

In [ ]:
from pathlib import Path
import json
import pandas as pd

out = Path("/kaggle/working/animalclef_salamander_eva_aliked_mergeonly_v20260430")
reports = out / "reports"
subs = out / "submissions"

summary_path = reports / "salamander_eva_aliked_mergeonly_v20260430_summary.json"
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))

print("\nBase shape:")
display(pd.read_csv(reports / "salamander_eva_aliked_mergeonly_v20260430_base_shape_report.csv"))

print("\nSelected rules:")
display(pd.read_csv(reports / "salamander_eva_aliked_mergeonly_v20260430_selected_rules.csv"))

print("\nAll shape reports:")
display(pd.read_csv(reports / "salamander_eva_aliked_mergeonly_v20260430_all_shape_reports.csv"))

print("\nActions by profile:")
for profile in ["ultra_p90", "strict_p90", "p95_gamble"]:
    action_path = reports / f"salamander_eva_aliked_mergeonly_v20260430_{profile}_actions.csv"
    actions = pd.read_csv(action_path)
    print(profile, "actions:", len(actions))
    display(actions.head(20))

submission = pd.read_csv(out / "submission.csv")
print("\nDefault candidate:")
print(out / "submission.csv")
print("rows:", len(submission))
print("max cluster label length:", submission["cluster"].astype(str).str.len().max())

print("\nOther candidate files:")
for path in sorted(subs.glob("*.csv")):
    print(path)
            

In [ ]:
from pathlib import Path
from IPython.display import Image as IPImage, display

viz = Path("/kaggle/working/animalclef_salamander_eva_aliked_mergeonly_v20260430/visualizations")
if viz.exists():
    shown = 0
    for path in sorted(viz.glob("*.jpg"))[:16]:
        print(path.name)
        display(IPImage(filename=str(path)))
        shown += 1
    if shown == 0:
        print("No merge visualizations because the safest profile made zero merges.")
else:
    print("No visualization directory written.")
            